In [2]:
import pandas as pd
delivery_df = pd.read_csv("../data/delivery_locations_full.csv")

import gurobipy as gp

n_orders = 50
n_possible_drones = 50
time_limit = 30  # minutes

df = delivery_df.iloc[:n_orders].reset_index(drop=True)
service_time = df["drone_unavailability_time"].tolist()

m = gp.Model("drone_delivery")

# x[i, d] = 1 if order i is assigned to drone d
x = m.addVars(n_orders, n_possible_drones,
              vtype=gp.GRB.BINARY, name="x")

# y[d] = 1 if drone d is used
y = m.addVars(n_possible_drones,
              vtype=gp.GRB.BINARY, name="y")

# Each order is assigned to exactly one drone
for i in range(n_orders):
    m.addConstr(
        gp.quicksum(x[i, d] for d in range(n_possible_drones)) == 1
    )

# A drone may only receive orders if it is activated
for i in range(n_orders):
    for d in range(n_possible_drones):
        m.addConstr(x[i, d] <= y[d])

# Total occupation time of each used drone is at most 30 minutes
for d in range(n_possible_drones):
    m.addConstr(
        gp.quicksum(x[i, d] * service_time[i] for i in range(n_orders))
        <= time_limit
    )

# Minimize required fleet size
m.setObjective(gp.quicksum(y[d] for d in range(n_possible_drones)),
               gp.GRB.MINIMIZE)

m.optimize()

print("Minimum number of drones:", round(m.ObjVal))

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2861913
Academic license 2861913 - for non-commercial use only - registered to ma___@kth.se
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2861913 - for non-commercial use only - registered to ma___@kth.se
Optimize a model with 2600 rows, 2550 columns and 10000 nonzeros
Model fingerprint: 0x12a73dda
Variable types: 0 continuous, 2550 integer (2550 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+01]
Found heuristic solution: objective 30.0000000
Presolve time: 0.01s
Presolved: 2600 rows, 2550 columns, 10000 nonzeros
Variable types: 0 continuous, 2550 integer (2550 binary)

Root relaxation: objective 1.000000e+00, 2529 iterations, 0.01 seconds